# Nexar Dashcam Collision Prediction

This notebook builds a temporal video classifier for the Nexar collision-prediction task. A pretrained ResNet18 extracts a feature vector from each sampled frame. A bidirectional GRU and temporal attention layer then process the ordered feature sequence.

The notebook creates its own metadata, video-level train/validation split, temporal clips, feature cache, model checkpoint, validation report, and Kaggle submission. It does not depend on any previous notebook outputs or external feature files.

Before running all cells, attach the Nexar Collision Prediction competition dataset in Kaggle, select a GPU accelerator, and enable Internet so the pretrained ImageNet weights can be loaded.


In [2]:
from pathlib import Path

data_root = Path("/kaggle/input/competitions/nexar-collision-prediction")

print("DATA ROOT EXISTS:", data_root.exists())

if data_root.exists():
    print("FILES:", sorted(p.name for p in data_root.iterdir()))
else:
    print("AVAILABLE TRAIN CSV PATHS:")
    for p in Path("/kaggle/input").rglob("train.csv"):
        print(p)

DATA ROOT EXISTS: True
FILES: ['sample_submission.csv', 'test', 'test.csv', 'train', 'train.csv']


In [3]:
import gc
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import resnet18, ResNet18_Weights
from tqdm.auto import tqdm

SEED = 42
DATA_ROOT = Path("/kaggle/input/competitions/nexar-collision-prediction")
WORK_ROOT = Path("/kaggle/working")
RUN_DIR = WORK_ROOT / "nexar_gru_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)

for item in RUN_DIR.iterdir():
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()

CLIP_DURATION = 10.0
HORIZONS = [0.5, 1.0, 1.5]
NUM_FRAMES = 24
IMAGE_HEIGHT = 128
IMAGE_WIDTH = 224
FRAME_BATCH_SIZE = 8 if torch.cuda.device_count() > 1 else 4
FRAME_NUM_WORKERS = 2
FEATURE_BATCH_SIZE = 64
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
cv2.setNumThreads(0)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("A Kaggle GPU accelerator is required. Select GPU in Notebook Settings before running all cells.")

print("Device:", device)
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Run directory:", RUN_DIR)


Device: cuda
GPU count: 2
GPU: Tesla T4
Run directory: /kaggle/working/nexar_gru_run


## 1. Load and validate the competition data

Video IDs are kept as strings so leading zeros are preserved. The input directory is read-only; all generated files are written to `/kaggle/working`.


In [4]:
required_paths = [
    DATA_ROOT / "train",
    DATA_ROOT / "test",
    DATA_ROOT / "train.csv",
    DATA_ROOT / "test.csv",
    DATA_ROOT / "sample_submission.csv"
]

missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Required competition files are missing: " + ", ".join(missing_paths))

train_df = pd.read_csv(DATA_ROOT / "train.csv", dtype={"id": str})
test_df = pd.read_csv(DATA_ROOT / "test.csv", dtype={"id": str})
sample_submission = pd.read_csv(DATA_ROOT / "sample_submission.csv", dtype={"id": str})

for frame in [train_df, test_df, sample_submission]:
    frame["id"] = frame["id"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)

train_df["target"] = train_df["target"].astype(int)

if train_df["id"].duplicated().any() or test_df["id"].duplicated().any():
    raise ValueError("Duplicate video IDs were found.")

if not set(train_df["target"].unique()).issubset({0, 1}):
    raise ValueError("The training target must contain only 0 and 1.")

if len(test_df) != len(sample_submission):
    raise ValueError("test.csv and sample_submission.csv have different row counts.")

if set(test_df["id"]) != set(sample_submission["id"]):
    raise ValueError("The test IDs do not match the sample submission IDs.")

missing_train_videos = [
    video_id for video_id in train_df["id"]
    if not (DATA_ROOT / "train" / f"{video_id}.mp4").is_file()
]
missing_test_videos = [
    video_id for video_id in test_df["id"]
    if not (DATA_ROOT / "test" / f"{video_id}.mp4").is_file()
]

if missing_train_videos or missing_test_videos:
    raise FileNotFoundError(
        f"Missing train videos: {len(missing_train_videos)}; "
        f"missing test videos: {len(missing_test_videos)}"
    )

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)
print("Target distribution:")
print(train_df["target"].value_counts().sort_index())
print("Train videos found:", len(train_df))
print("Test videos found:", len(test_df))


Train shape: (1500, 4)
Test shape: (1344, 1)
Sample submission shape: (1344, 2)
Target distribution:
target
0    750
1    750
Name: count, dtype: int64
Train videos found: 1500
Test videos found: 1344


## 2. Extract video metadata

The duration and frame rate are read directly from each MP4 file. This step is performed once and the resulting metadata is saved in the run directory.


In [5]:
def read_video_metadata(video_path, video_id, split_name):
    capture = cv2.VideoCapture(str(video_path))
    opened = capture.isOpened()

    if not opened:
        capture.release()
        raise RuntimeError(f"Could not open {video_path}")

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()

    if not np.isfinite(fps) or fps <= 0 or frame_count <= 0:
        raise RuntimeError(f"Invalid video metadata for {video_path}")

    return {
        "id": video_id,
        "split": split_name,
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_sec": frame_count / fps
    }


def collect_video_metadata(frame, split_name):
    records = []
    video_dir = DATA_ROOT / split_name

    for video_id in tqdm(frame["id"].tolist(), desc=f"Reading {split_name} metadata"):
        video_path = video_dir / f"{video_id}.mp4"
        records.append(read_video_metadata(video_path, video_id, split_name))

    return pd.DataFrame(records)


train_meta = collect_video_metadata(train_df, "train")
test_meta = collect_video_metadata(test_df, "test")
video_meta = pd.concat([train_meta, test_meta], ignore_index=True)

video_meta.to_csv(RUN_DIR / "video_metadata.csv", index=False)

train_data = train_df.merge(
    train_meta[["id", "fps", "frame_count", "duration_sec"]],
    on="id",
    how="left",
    validate="one_to_one"
)

test_data = test_df.merge(
    test_meta[["id", "fps", "frame_count", "duration_sec"]],
    on="id",
    how="left",
    validate="one_to_one"
)

if train_data["duration_sec"].isna().any() or test_data["duration_sec"].isna().any():
    raise ValueError("Video metadata could not be aligned with the CSV files.")

print("Metadata shape:", video_meta.shape)
print(video_meta.groupby("split")["duration_sec"].agg(["min", "median", "mean", "max"]).round(3))
print("Unreadable videos: 0")


Reading train metadata:   0%|          | 0/1500 [00:00<?, ?it/s]

Reading test metadata:   0%|          | 0/1344 [00:00<?, ?it/s]

Metadata shape: (2844, 7)
          min  median    mean     max
split                                
test    6.567   9.960   9.871  11.333
train  14.000  40.131  37.741  60.000
Unreadable videos: 0


## 3. Create a video-level split and temporal training clips

The split is performed before clip generation, so clips from the same source video cannot appear in both training and validation.

For a positive video, each training example ends 0.5, 1.0, or 1.5 seconds before its annotated event. Its temporal label is positive only when the alert time has already been reached at the clip endpoint. Negative videos contribute three negative clips at matched temporal positions. The negative clip-position distribution is estimated from the training fold only.


In [6]:
train_fold, val_fold = train_test_split(
    train_data,
    test_size=0.2,
    random_state=SEED,
    stratify=train_data["target"]
)

train_fold = train_fold.reset_index(drop=True)
val_fold = val_fold.reset_index(drop=True)

positive_train = train_fold.loc[train_fold["target"] == 1].copy()
positive_train["event_offset"] = (
    positive_train["time_of_event"] - positive_train["duration_sec"] / 2
)

event_offsets = positive_train["event_offset"].dropna().to_numpy(dtype=np.float64)
if len(event_offsets) < 10:
    raise ValueError("Not enough positive training videos to estimate temporal offsets.")

offset_low, offset_high = np.quantile(event_offsets, [0.02, 0.98])
event_offsets = np.clip(event_offsets, offset_low, offset_high)
EVENT_OFFSET_MEAN = float(event_offsets.mean())
EVENT_OFFSET_STD = float(event_offsets.std(ddof=1))
EVENT_OFFSET_STD = max(EVENT_OFFSET_STD, 0.5)


def build_temporal_clips(frame, seed):
    rng = np.random.default_rng(seed)
    records = []

    for row in frame.itertuples(index=False):
        video_id = str(row.id).zfill(5)
        duration = float(row.duration_sec)
        video_path = str(DATA_ROOT / "train" / f"{video_id}.mp4")
        target = int(row.target)

        if target == 1:
            event_time = float(row.time_of_event)
            alert_time = float(row.time_of_alert)

            if not (0 < alert_time <= event_time <= duration):
                raise ValueError(f"Invalid event timestamps for video {video_id}")

            reference_event_time = event_time
        else:
            offset = float(rng.normal(EVENT_OFFSET_MEAN, EVENT_OFFSET_STD))
            reference_event_time = duration / 2 + offset

        for horizon in HORIZONS:
            if target == 1:
                clip_end = reference_event_time - horizon
                if clip_end <= 0:
                    raise ValueError(f"Invalid temporal clip endpoint for video {video_id}")
                label = int((event_time - alert_time) >= horizon)
            else:
                minimum_end = min(CLIP_DURATION, duration)
                clip_end = float(np.clip(reference_event_time - horizon, minimum_end, duration))
                label = 0

            clip_start = max(0.0, clip_end - CLIP_DURATION)
            clip_duration = clip_end - clip_start

            if clip_duration <= 0:
                raise ValueError(f"Invalid clip duration for video {video_id}")

            records.append({
                "id": video_id,
                "video_path": video_path,
                "clip_start": float(clip_start),
                "clip_end": float(clip_end),
                "clip_duration": float(clip_duration),
                "horizon": float(horizon),
                "label": int(label),
                "video_target": int(target),
                "duration_sec": duration
            })

    clips = pd.DataFrame(records)
    expected_rows = len(frame) * len(HORIZONS)

    if len(clips) != expected_rows:
        raise ValueError(f"{split_name} clip count mismatch: {len(clips)} vs {expected_rows}")

    if clips.duplicated(["id", "horizon"]).any():
        raise ValueError(f"Duplicate video-horizon pairs in {split_name} clips.")

    if not clips.groupby("id").size().eq(len(HORIZONS)).all():
        raise ValueError("Some videos do not have all temporal horizons.")

    for horizon in HORIZONS:
        positive_source = frame.loc[frame["target"] == 1]
        expected_positive = int((positive_source["time_of_event"] - positive_source["time_of_alert"] >= horizon).sum())
        horizon_rows = clips[np.isclose(clips["horizon"], horizon)]
        actual_positive = int(horizon_rows.loc[horizon_rows["video_target"] == 1, "label"].sum())
        if expected_positive != actual_positive:
            raise ValueError(f"Temporal label audit failed at horizon {horizon}.")

    return clips


train_clips = build_temporal_clips(train_fold, SEED + 1)
val_clips = build_temporal_clips(val_fold, SEED + 2)

train_overlap = set(train_clips["id"]) & set(val_clips["id"])
if train_overlap:
    raise ValueError(f"Video leakage detected: {len(train_overlap)} overlapping IDs")

for split_name, clips in [("Train", train_clips), ("Validation", val_clips)]:
    print(f"\n{split_name} clips:", len(clips))
    print(pd.crosstab(clips["horizon"], clips["label"]))

train_clips.to_csv(RUN_DIR / "train_clips.csv", index=False)
val_clips.to_csv(RUN_DIR / "val_clips.csv", index=False)

print("Training videos:", len(train_fold))
print("Validation videos:", len(val_fold))
print("Video overlap:", len(train_overlap))
print("Event offset mean/std:", round(EVENT_OFFSET_MEAN, 4), round(EVENT_OFFSET_STD, 4))



Train clips: 3600
label      0    1
horizon          
0.5      638  562
1.0      743  457
1.5      915  285

Validation clips: 900
label      0    1
horizon          
0.5      159  141
1.0      196  104
1.5      232   68
Training videos: 1200
Validation videos: 300
Video overlap: 0
Event offset mean/std: -0.0944 0.9086


## 4. Build the video-frame dataset

Each clip is sampled at 24 evenly spaced timestamps in chronological order. Frames are resized to 128×224, preserving the original 16:9 video aspect ratio, and returned as uint8 tensors to reduce host-memory and transfer costs.


In [7]:
class DashcamClipDataset(Dataset):
    def __init__(self, clips, num_frames=NUM_FRAMES, image_height=IMAGE_HEIGHT, image_width=IMAGE_WIDTH):
        self.clips = clips.reset_index(drop=True).copy()
        self.num_frames = int(num_frames)
        self.image_height = int(image_height)
        self.image_width = int(image_width)

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, index):
        row = self.clips.iloc[index]
        video_path = Path(row["video_path"])
        capture = cv2.VideoCapture(str(video_path))

        if not capture.isOpened():
            capture.release()
            raise RuntimeError(f"Could not open video {video_path}")

        try:
            fps = float(capture.get(cv2.CAP_PROP_FPS))
            total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

            if fps <= 0 or total_frames <= 0:
                raise RuntimeError(f"Invalid video properties for {video_path}")

            start_frame = int(np.floor(float(row["clip_start"]) * fps))
            end_frame = int(np.floor(float(row["clip_end"]) * fps))
            start_frame = int(np.clip(start_frame, 0, total_frames - 1))
            end_frame = int(np.clip(end_frame, start_frame, total_frames - 1))

            frame_indices = np.rint(
                np.linspace(start_frame, end_frame, self.num_frames)
            ).astype(np.int64)
            selected_indices = set(frame_indices.tolist())
            frames_by_index = {}

            capture.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

            for frame_index in range(start_frame, end_frame + 1):
                success, frame = capture.read()
                if not success:
                    break

                if frame_index in selected_indices:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = cv2.resize(
                        frame,
                        (self.image_width, self.image_height),
                        interpolation=cv2.INTER_AREA
                    )
                    frames_by_index[frame_index] = frame

            if not frames_by_index:
                raise RuntimeError(f"No frames could be decoded from {video_path}")

            available_indices = np.array(sorted(frames_by_index.keys()), dtype=np.int64)
            frames = []

            for frame_index in frame_indices:
                if int(frame_index) in frames_by_index:
                    frames.append(frames_by_index[int(frame_index)])
                else:
                    nearest_index = int(available_indices[np.argmin(np.abs(available_indices - frame_index))])
                    frames.append(frames_by_index[nearest_index])

        finally:
            capture.release()

        frames = np.stack(frames, axis=0)
        frames = np.ascontiguousarray(frames.transpose(0, 3, 1, 2))
        frame_tensor = torch.from_numpy(frames)
        label = torch.tensor(float(row.get("label", -1)), dtype=torch.float32)
        horizon = torch.tensor(float(row.get("horizon", -1)), dtype=torch.float32)

        return frame_tensor, label, str(row["id"]), horizon


train_frame_dataset = DashcamClipDataset(train_clips)
val_frame_dataset = DashcamClipDataset(val_clips)

train_frame_loader = DataLoader(
    train_frame_dataset,
    batch_size=FRAME_BATCH_SIZE,
    shuffle=False,
    num_workers=FRAME_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=FRAME_NUM_WORKERS > 0
)

val_frame_loader = DataLoader(
    val_frame_dataset,
    batch_size=FRAME_BATCH_SIZE,
    shuffle=False,
    num_workers=FRAME_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=FRAME_NUM_WORKERS > 0
)

sample_frames, sample_label, sample_id, sample_horizon = train_frame_dataset[0]
print("Frame tensor shape:", tuple(sample_frames.shape))
print("Frame dtype:", sample_frames.dtype)
print("Sample ID:", sample_id)
print("Sample label/horizon:", sample_label.item(), sample_horizon.item())

if sample_frames.shape != (NUM_FRAMES, 3, IMAGE_HEIGHT, IMAGE_WIDTH):
    raise ValueError("Unexpected frame tensor shape.")


Frame tensor shape: (24, 3, 128, 224)
Frame dtype: torch.uint8
Sample ID: 00780
Sample label/horizon: 1.0 0.5


## 5. Create the pretrained frame encoder

ResNet18 produces one 512-dimensional visual descriptor per frame. Its parameters remain frozen during this stage. The ordered sequence of descriptors is passed to the GRU in a later section.


In [8]:
weights = ResNet18_Weights.DEFAULT
backbone = resnet18(weights=weights)
frame_encoder = nn.Sequential(*list(backbone.children())[:-1])

for parameter in frame_encoder.parameters():
    parameter.requires_grad = False

frame_encoder = frame_encoder.to(device)

if torch.cuda.device_count() > 1:
    frame_encoder = nn.DataParallel(
        frame_encoder,
        device_ids=list(range(torch.cuda.device_count()))
    )

frame_encoder.eval()
encoder_to_save = frame_encoder.module if isinstance(frame_encoder, nn.DataParallel) else frame_encoder
torch.save(encoder_to_save.state_dict(), RUN_DIR / "resnet18_frame_encoder.pth")

print("Frame encoder:", type(backbone).__name__)
print("Feature dimension:", backbone.fc.in_features)
print("Pretrained weights:", weights.__class__.__name__)
print("Device:", device)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s] 


Frame encoder: ResNet
Feature dimension: 512
Pretrained weights: ResNet18_Weights
Device: cuda


## 6. Extract and cache ordered frame features

Feature extraction is performed once for each split. Training clips receive mild clip-consistent brightness and contrast augmentation. Validation and test clips are processed deterministically. Each saved example retains its full `(time, feature)` sequence; no temporal mean pooling is performed.


In [9]:
imagenet_mean = torch.tensor(
    [0.485, 0.456, 0.406],
    device=device
).view(1, 1, 3, 1, 1)

imagenet_std = torch.tensor(
    [0.229, 0.224, 0.225],
    device=device
).view(1, 1, 3, 1, 1)


def extract_feature_sequences(loader, split_name, augment=False):
    feature_batches = []
    label_batches = []
    horizon_batches = []
    video_ids = []
    frame_encoder.eval()

    for frames, labels, ids, horizons in tqdm(loader, desc=f"Extracting {split_name} features"):
        frames = frames.to(device, non_blocking=True).float().div_(255.0)

        if augment:
            brightness = torch.empty(
                (frames.shape[0], 1, 1, 1, 1),
                device=device
            ).uniform_(0.90, 1.10)
            contrast = torch.empty(
                (frames.shape[0], 1, 1, 1, 1),
                device=device
            ).uniform_(0.90, 1.10)
            clip_mean = frames.mean(dim=(1, 2, 3, 4), keepdim=True)
            frames = (frames - clip_mean) * contrast + clip_mean
            frames = (frames * brightness).clamp_(0.0, 1.0)

        frames = (frames - imagenet_mean) / imagenet_std
        batch_size, time_steps, channels, height, width = frames.shape
        flat_frames = frames.reshape(batch_size * time_steps, channels, height, width)

        with torch.inference_mode():
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=device.type == "cuda"
            ):
                encoded = frame_encoder(flat_frames)

        encoded = encoded.float().flatten(1)
        encoded = encoded.reshape(batch_size, time_steps, -1).cpu()

        feature_batches.append(encoded)
        label_batches.append(labels.cpu())
        horizon_batches.append(horizons.cpu())
        video_ids.extend(list(ids))

    result = {
        "features": torch.cat(feature_batches, dim=0).contiguous(),
        "labels": torch.cat(label_batches, dim=0).float(),
        "horizons": torch.cat(horizon_batches, dim=0).float(),
        "ids": video_ids
    }

    if result["features"].ndim != 3:
        raise ValueError(f"Invalid feature tensor shape for {split_name}")

    if result["features"].shape[1] != NUM_FRAMES:
        raise ValueError(f"Temporal dimension mismatch for {split_name}")

    if not torch.isfinite(result["features"]).all():
        raise ValueError(f"Non-finite frame features found in {split_name}")

    torch.save(result, RUN_DIR / f"{split_name.lower()}_features.pt")
    print(
        split_name,
        "features:",
        tuple(result["features"].shape),
        "labels:",
        tuple(result["labels"].shape)
    )

    return result


train_cache = extract_feature_sequences(train_frame_loader, "Train", augment=True)
val_cache = extract_feature_sequences(val_frame_loader, "Validation", augment=False)

del train_frame_loader, val_frame_loader
_ = gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Extracting Train features:   0%|          | 0/450 [00:00<?, ?it/s]

Train features: (3600, 24, 512) labels: (3600,)


Extracting Validation features:   0%|          | 0/113 [00:00<?, ?it/s]

Validation features: (900, 24, 512) labels: (900,)


## 7. Define the temporal GRU classifier

The GRU reads the frame descriptors in chronological order. A learned attention layer assigns different weights to the GRU outputs, allowing the classifier to focus on informative moments instead of averaging frame features uniformly.


In [10]:
class TemporalGRUClassifier(nn.Module):
    def __init__(self, input_size=512, hidden_size=256, num_layers=2, dropout=0.30):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        representation_size = hidden_size * 2
        self.attention = nn.Sequential(
            nn.Linear(representation_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        self.normalization = nn.LayerNorm(representation_size)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(representation_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.75),
            nn.Linear(128, 1)
        )

    def forward(self, sequences):
        sequence_outputs, _ = self.gru(sequences)
        attention_scores = self.attention(sequence_outputs)
        attention_weights = torch.softmax(attention_scores, dim=1)
        context = torch.sum(sequence_outputs * attention_weights, dim=1)
        context = self.normalization(context)
        return self.classifier(context).squeeze(1)


input_size = int(train_cache["features"].shape[-1])
model = TemporalGRUClassifier(input_size=input_size).to(device)

train_feature_dataset = TensorDataset(
    train_cache["features"],
    train_cache["labels"],
    train_cache["horizons"]
)

val_feature_dataset = TensorDataset(
    val_cache["features"],
    val_cache["labels"],
    val_cache["horizons"]
)

train_loader = DataLoader(
    train_feature_dataset,
    batch_size=FEATURE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_feature_dataset,
    batch_size=FEATURE_BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

positive_count = float(train_cache["labels"].sum().item())
negative_count = float(len(train_cache["labels"]) - positive_count)
if positive_count == 0 or negative_count == 0:
    raise ValueError("Training labels must contain both classes.")

positive_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    threshold=0.0005,
    min_lr=1e-5
)

print("Input feature shape:", tuple(train_cache["features"].shape))
print("Trainable GRU parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Positive weight:", round(positive_weight.item(), 4))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Input feature shape: (3600, 24, 512)
Trainable GRU parameters: 2498050
Positive weight: 1.7607
Train batches: 57
Validation batches: 8


## 8. Train with horizon-wise validation and early stopping

The primary validation metric is mean Average Precision across the 0.5s, 1.0s, and 1.5s horizons. The best checkpoint is selected using this metric. Gradient clipping, AdamW weight decay, dropout, learning-rate reduction, and early stopping are used to control overfitting.


In [11]:
def calculate_horizon_metrics(labels, probabilities, horizons):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities).astype(float)
    horizons = np.asarray(horizons).astype(float)
    rows = []

    for horizon in HORIZONS:
        mask = np.isclose(horizons, horizon)
        y_true = labels[mask]
        y_score = probabilities[mask]

        if len(y_true) == 0 or len(np.unique(y_true)) < 2:
            ap = np.nan
            auc = np.nan
        else:
            ap = float(average_precision_score(y_true, y_score))
            auc = float(roc_auc_score(y_true, y_score))

        rows.append({
            "horizon": float(horizon),
            "average_precision": ap,
            "roc_auc": auc,
            "positive_rate": float(y_true.mean()) if len(y_true) else np.nan,
            "samples": int(len(y_true))
        })

    metrics = pd.DataFrame(rows)
    mean_ap = float(metrics["average_precision"].mean())
    return metrics, mean_ap


def predict_feature_dataset(loader):
    model.eval()
    all_labels = []
    all_probabilities = []
    all_horizons = []

    with torch.inference_mode():
        for features, labels, horizons in loader:
            features = features.to(device, non_blocking=True)
            logits = model(features)
            probabilities = torch.sigmoid(logits.float())
            all_labels.extend(labels.numpy().tolist())
            all_probabilities.extend(probabilities.cpu().numpy().tolist())
            all_horizons.extend(horizons.numpy().tolist())

    return (
        np.asarray(all_labels, dtype=np.int64),
        np.asarray(all_probabilities, dtype=np.float64),
        np.asarray(all_horizons, dtype=np.float64)
    )


best_val_map = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []
CHECKPOINT_PATH = RUN_DIR / "best_temporal_gru.pth"

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    sample_count = 0

    for features, labels, horizons in tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS}"):
        features = features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += float(loss.item()) * batch_size
        sample_count += batch_size

    train_loss = running_loss / max(sample_count, 1)
    val_labels, val_probabilities, val_horizons = predict_feature_dataset(val_loader)
    val_metrics, val_map = calculate_horizon_metrics(
        val_labels,
        val_probabilities,
        val_horizons
    )
    val_loss_total = 0.0
    val_count = 0

    model.eval()
    with torch.inference_mode():
        for features, labels, horizons in val_loader:
            features = features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()
            loss = criterion(model(features), labels)
            val_loss_total += float(loss.item()) * labels.size(0)
            val_count += labels.size(0)

    val_loss = val_loss_total / max(val_count, 1)
    scheduler.step(val_map)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "validation_map": val_map,
        "learning_rate": optimizer.param_groups[0]["lr"]
    }
    history.append(row)

    print("\nEpoch:", epoch)
    print("Train loss:", round(train_loss, 5))
    print("Validation loss:", round(val_loss, 5))
    print(val_metrics.round(4).to_string(index=False))
    print("Validation mAP:", round(val_map, 5))

    if np.isfinite(val_map) and val_map > best_val_map + 1e-5:
        best_val_map = val_map
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "best_val_map": best_val_map,
                "best_epoch": best_epoch,
                "input_size": input_size,
                "num_frames": NUM_FRAMES,
                "image_height": IMAGE_HEIGHT,
                "image_width": IMAGE_WIDTH,
                "horizons": HORIZONS
            },
            CHECKPOINT_PATH
        )
        print("Best checkpoint saved.")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping activated.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / "training_history.csv", index=False)

if not CHECKPOINT_PATH.exists():
    raise RuntimeError("Training finished without producing a valid checkpoint.")

print("\nBest epoch:", best_epoch)
print("Best validation mAP:", round(best_val_map, 5))
print("Checkpoint:", CHECKPOINT_PATH)


Epoch 1/30:   0%|          | 0/57 [00:00<?, ?it/s]


Epoch: 1
Train loss: 0.84822
Validation loss: 0.80997
 horizon  average_precision  roc_auc  positive_rate  samples
     0.5             0.7111   0.7423         0.4700      300
     1.0             0.5365   0.7126         0.3467      300
     1.5             0.4143   0.7234         0.2267      300
Validation mAP: 0.55397
Best checkpoint saved.


Epoch 2/30:   0%|          | 0/57 [00:00<?, ?it/s]


Epoch: 2
Train loss: 0.7682
Validation loss: 0.79209
 horizon  average_precision  roc_auc  positive_rate  samples
     0.5             0.7389   0.7591         0.4700      300
     1.0             0.5896   0.7354         0.3467      300
     1.5             0.4754   0.7561         0.2267      300
Validation mAP: 0.6013
Best checkpoint saved.


Epoch 3/30:   0%|          | 0/57 [00:00<?, ?it/s]


Epoch: 3
Train loss: 0.75591
Validation loss: 0.79654
 horizon  average_precision  roc_auc  positive_rate  samples
     0.5             0.7534   0.7715         0.4700      300
     1.0             0.6011   0.7460         0.3467      300
     1.5             0.4806   0.7672         0.2267      300
Validation mAP: 0.61172
Best checkpoint saved.


... (12 outputs left out)

Epoch 10/30:   0%|          | 0/57 [00:00<?, ?it/s]


Epoch: 10
Train loss: 0.47695
Validation loss: 0.96943
 horizon  average_precision  roc_auc  positive_rate  samples
     0.5             0.7431   0.7606         0.4700      300
     1.0             0.5676   0.7251         0.3467      300
     1.5             0.4389   0.7208         0.2267      300
Validation mAP: 0.58323


Epoch 11/30:   0%|          | 0/57 [00:00<?, ?it/s]


Epoch: 11
Train loss: 0.42174
Validation loss: 1.16057
 horizon  average_precision  roc_auc  positive_rate  samples
     0.5             0.7165   0.7336         0.4700      300
     1.0             0.5479   0.7084         0.3467      300
     1.5             0.4459   0.7273         0.2267      300
Validation mAP: 0.57007
Early stopping activated.

Best epoch: 5
Best validation mAP: 0.61823
Checkpoint: /kaggle/working/nexar_gru_run/best_temporal_gru.pth


## 9. Restore and evaluate the best checkpoint

The best checkpoint is reloaded before the final validation report. The report includes Average Precision and ROC-AUC separately for each temporal horizon.


In [12]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

val_labels, val_probabilities, val_horizons = predict_feature_dataset(val_loader)
final_val_metrics, final_val_map = calculate_horizon_metrics(
    val_labels,
    val_probabilities,
    val_horizons
)

final_val_metrics.to_csv(RUN_DIR / "validation_metrics.csv", index=False)

print(final_val_metrics.round(4).to_string(index=False))
print("Final validation mAP:", round(final_val_map, 5))
print("Best checkpoint epoch:", checkpoint["best_epoch"])

if not np.isfinite(final_val_map):
    raise ValueError("Validation mAP is not finite.")


 horizon  average_precision  roc_auc  positive_rate  samples
     0.5             0.7541   0.7686         0.4700      300
     1.0             0.6092   0.7436         0.3467      300
     1.5             0.4914   0.7629         0.2267      300
Final validation mAP: 0.61823
Best checkpoint epoch: 5


## 10. Prepare one test clip per video

Each test video is approximately ten seconds long and ends at one of the hidden anticipation horizons. We therefore use the final ten seconds of each test video, preserving the endpoint of the supplied clip, and generate one probability per video.


In [ ]:
test_records = []

for row in test_data.itertuples(index=False):
    video_id = str(row.id).zfill(5)
    duration = float(row.duration_sec)
    clip_end = duration
    clip_start = max(0.0, clip_end - CLIP_DURATION)
    video_path = DATA_ROOT / "test" / f"{video_id}.mp4"

    test_records.append({
        "id": video_id,
        "video_path": str(video_path),
        "clip_start": float(clip_start),
        "clip_end": float(clip_end),
        "clip_duration": float(clip_end - clip_start),
        "horizon": -1.0,
        "label": -1,
        "video_target": -1,
        "duration_sec": duration
    })

test_clips = pd.DataFrame(test_records)

if len(test_clips) != len(test_df) or test_clips["id"].nunique() != len(test_df):
    raise ValueError("Test clip construction did not produce exactly one clip per video.")

if test_clips["clip_duration"].le(0).any():
    raise ValueError("A test clip has a non-positive duration.")

test_clips.to_csv(RUN_DIR / "test_clips.csv", index=False)

test_frame_dataset = DashcamClipDataset(test_clips)
test_frame_loader = DataLoader(
    test_frame_dataset,
    batch_size=FRAME_BATCH_SIZE,
    shuffle=False,
    num_workers=FRAME_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=FRAME_NUM_WORKERS > 0
)

print("Test videos:", len(test_df))
print("Test clips:", len(test_clips))
print("Test clip duration range:")
print(test_clips["clip_duration"].agg(["min", "median", "max"]).round(3))


Test videos: 1344
Test clips: 1344
Test clip duration range:
min        6.567
median     9.960
max       10.000
Name: clip_duration, dtype: float64


## 11. Extract test features and generate predictions

The test feature sequences are extracted with the same frame encoder and normalization used for validation. The GRU converts each ordered sequence into a probability score.


In [ ]:
test_cache = extract_feature_sequences(
    test_frame_loader,
    "Test",
    augment=False
)

del test_frame_loader, test_frame_dataset
_ = gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

test_feature_dataset = TensorDataset(test_cache["features"])
test_feature_loader = DataLoader(
    test_feature_dataset,
    batch_size=FEATURE_BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

model.eval()
test_probabilities = []

with torch.inference_mode():
    for (features,) in tqdm(test_feature_loader, desc="Predicting test videos"):
        features = features.to(device, non_blocking=True)
        probabilities = torch.sigmoid(model(features).float())
        test_probabilities.extend(probabilities.cpu().numpy().tolist())

test_predictions = pd.DataFrame({
    "id": test_cache["ids"],
    "target": np.asarray(test_probabilities, dtype=np.float64)
})

test_predictions["id"] = test_predictions["id"].astype(str).str.zfill(5)

if len(test_predictions) != len(test_df):
    raise ValueError("The number of predictions does not match the test set.")

if test_predictions["id"].duplicated().any():
    raise ValueError("Duplicate test predictions were generated.")

if not np.isfinite(test_predictions["target"]).all():
    raise ValueError("Non-finite test probabilities were generated.")

if not test_predictions["target"].between(0.0, 1.0).all():
    raise ValueError("Test probabilities must be between 0 and 1.")

print("Predictions generated:", len(test_predictions))
print(test_predictions.head(10).to_string(index=False))


Extracting Test features:   0%|          | 0/168 [00:00<?, ?it/s]

Test features: (1344, 24, 512) labels: (1344,)


Predicting test videos:   0%|          | 0/11 [00:00<?, ?it/s]

Predictions generated: 1344
   id   target
00204 0.345563
00030 0.514452
00146 0.547404
00020 0.491676
00511 0.800724
00261 0.615506
00346 0.684136
00545 0.646740
00492 0.400625
00299 0.734499


## 12. Create and validate the Kaggle submission

The final file follows the competition sample-submission column order and retains the original test-video order. The output is written to `/kaggle/working/submission.csv`.


In [ ]:
submission = sample_submission[["id"]].copy()
submission["id"] = submission["id"].astype(str).str.zfill(5)

submission = submission.merge(
    test_predictions,
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

submission = submission[["id", "target"]]

if len(submission) != len(sample_submission):
    raise ValueError("Submission row count does not match the sample file.")

if submission["id"].duplicated().any():
    raise ValueError("Submission contains duplicate IDs.")

if submission["target"].isna().any():
    raise ValueError("Submission contains missing prediction scores.")

if not np.isfinite(submission["target"]).all():
    raise ValueError("Submission contains non-finite scores.")

if not submission["target"].between(0.0, 1.0).all():
    raise ValueError("Submission scores must be between 0 and 1.")

if submission["id"].tolist() != sample_submission["id"].astype(str).str.zfill(5).tolist():
    raise ValueError("Submission row order differs from sample_submission.csv.")

submission_path = WORK_ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)
submission.to_csv(RUN_DIR / "submission.csv", index=False)

print("Submission created:", submission_path)
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("Score range:", round(float(submission["target"].min()), 6), round(float(submission["target"].max()), 6))
print("Missing values:", submission.isna().sum().to_dict())
print(submission.head(10).to_string(index=False))
print("\nRUN ALL COMPLETED SUCCESSFULLY.")


Submission created: /kaggle/working/submission.csv
Shape: (1344, 2)
Columns: ['id', 'target']
Score range: 0.011073 0.854871
Missing values: {'id': 0, 'target': 0}
   id   target
00204 0.345563
00030 0.514452
00146 0.547404
00020 0.491676
00511 0.800724
00261 0.615506
00346 0.684136
00545 0.646740
00492 0.400625
00299 0.734499

RUN ALL COMPLETED SUCCESSFULLY.


## Output files

The main deliverable for Kaggle is `/kaggle/working/submission.csv`. The `nexar_gru_run` folder also contains the best GRU checkpoint, video metadata, temporal clip tables, cached feature sequences, training history, and validation metrics.
